# AsyncFlow — MMc Theory vs Simulation (Guided Notebook)

This notebook shows how to:

1. Make imports work inside a notebook (src-layout or package install)
2. Build a **multi-server** scenario compatible with **M/M/c** assumptions
3. Run the simulation and collect results
4. Compare theory vs observed KPIs (pretty-printed table)
5. Plot the standard dashboards (latency, throughput, server time series)

> Tip: run this notebook from your project **root folder**.


In [75]:
import importlib, asyncflow, asyncflow.analysis
importlib.reload(asyncflow)
importlib.reload(asyncflow.analysis)

<module 'asyncflow.analysis' from '/home/gioele/projects/AsyncFlow/src/asyncflow/analysis/__init__.py'>

In [76]:
import matplotlib.pyplot as plt
import simpy

# Public AsyncFlow API
from asyncflow import AsyncFlow, SimulationRunner, Sweep
from asyncflow.components import Client, Server, Edge, Endpoint, LoadBalancer
from asyncflow.settings import SimulationSettings
from asyncflow.workload import RqsGenerator
from asyncflow.analysis import MM1, ResultsAnalyzer, SweepAnalyzer, MMc

print("Imports OK.")

Imports OK.


## 1) Build an M/M/c-friendly scenario

* **Multiple identical servers with exponential CPU service**
  Topology includes **\$c \geq 2\$ identical servers**, each exposing exactly **one endpoint** with exactly **one CPU-bound step**.
  Service times follow an **Exponential** distribution with mean \$E\[S]\$ (service rate \$\mu = 1/E\[S]\$). No RAM/IO steps are included in the pipeline.

* **Load balancer with round-robin dispatch**
  A **single load balancer** is required when \$c > 1\$. It splits arrivals **round-robin** across servers, so each server has its own local queue.
  This corresponds to a **split M/M/c** model, not the textbook pooled queue.

* **Deterministic, very small network latency**
  All edges have **fixed latency** \$\ll 1,\mathrm{ms}\$. Queueing behavior is therefore dominated by CPU service, closely matching textbook assumptions.

* **“Poisson arrivals” via the generator**
  Arrivals are produced by the same **two-stage, windowed Poisson sampler**: in each user-sampling window \$\Delta\$, we draw the active users \$U\$ (Poisson or Normal, per config).
  Within the window, arrivals are a **homogeneous Poisson process** with rate \$\Lambda = U \cdot \lambda\_r/60\$.

  With **small \$\Delta\$**, **Poisson users**, **long runs**, and **tiny edge latency**, the aggregate arrivals seen by the load balancer approximate a global Poisson input, yielding a good empirical match to the M/M/c model.

---

```mermaid
graph LR;
    rqs1["<b>RqsGenerator</b><br/>id: rqs-1"]
    client1["<b>Client</b><br/>id: client-1"]
    lb1["<b>LoadBalancer</b><br/>id: lb-1<br/>Policy: round_robin"]
    app1["<b>Server</b><br/>id: app-1<br/>Endpoint: /api"]
    app2["<b>Server</b><br/>id: app-2<br/>Endpoint: /api"]

    rqs1 -- "Edge: gen-client<br/>Latency: 0.0001" --> client1;
    client1 -- "Request<br/>Edge: client-lb<br/>Latency: 0.0001" --> lb1;
    lb1 -- "Dispatch<br/>Edge: lb-app1<br/>Latency: 0.0001" --> app1;
    lb1 -- "Dispatch<br/>Edge: lb-app2<br/>Latency: 0.0001" --> app2;
    app1 -- "Response<br/>Edge: app1-client<br/>Latency: 0.0001" --> client1;
    app2 -- "Response<br/>Edge: app2-client<br/>Latency: 0.0001" --> client1;
```

---

⚠️ **Note on model scope**
This scenario currently represents a **split M/M/c with random dispatch**.
The **textbook M/M/c (Erlang-C)** assumes a **single pooled FCFS queue feeding c servers**, which tends to give lower waiting times (no imbalance across local queues).

In a future step, we will extend AsyncFlow with a **pooled FCFS dispatcher** at the load balancer, enabling direct comparison against the textbook Erlang-C closed forms.

---



In [ ]:
def build_payload():
    generator = RqsGenerator(
        id="rqs-1",
        avg_active_users={"mean": 120},
        avg_request_per_minute_per_user={"mean": 20},
        user_sampling_window=60,
    )

    client = Client(id="client-1")

    endpoint = Endpoint(
        endpoint_name="/api",
        probability=1.0,
        steps=[
            {
                "kind": "initial_parsing",
                "step_operation": {
                    "cpu_time": {"mean": 0.01, "distribution": "exponential"},
                },
            },
        ],
    )

    srv1 = Server(
        id="srv-1",
        server_resources={"cpu_cores": 1, "ram_mb": 2048},
        endpoints=[endpoint],
    )
    srv2 = Server(
        id="srv-2",
        server_resources={"cpu_cores": 1, "ram_mb": 2048},
        endpoints=[endpoint],
    )

    lb = LoadBalancer(
        id="lb-1",
        algorithms="random",  
        server_covered={"srv-1", "srv-2"},
    )

    edges = [
        Edge(id="gen-client",  source="rqs-1",  target="client-1", latency=0.00001, dropout_rate=0),
        Edge(id="client-lb",   source="client-1", target="lb-1",   latency=0.00001, dropout_rate=0),
        Edge(id="lb-srv1",     source="lb-1",   target="srv-1",    latency=0.00001, dropout_rate=0),
        Edge(id="lb-srv2",     source="lb-1",   target="srv-2",    latency=0.00001, dropout_rate=0),
        Edge(id="srv1-client", source="srv-1",  target="client-1", latency=0.00001, dropout_rate=0),
        Edge(id="srv2-client", source="srv-2",  target="client-1", latency=0.00001, dropout_rate=0),
    ]

    settings = SimulationSettings(
        total_simulation_time=2400,
        sample_period_s=0.05,
    )

    payload = (
        AsyncFlow()
        .add_generator(generator)
        .add_client(client)
        .add_servers(srv1, srv2)
        .add_load_balancer(lb)
        .add_edges(*edges)
        .add_simulation_settings(settings)
    ).build_payload()

    return payload


## 2) Run the simulation


In [78]:
payload = build_payload()
env = simpy.Environment()
runner = SimulationRunner(env=env, simulation_input=payload)
results: ResultsAnalyzer = runner.run()
print("Done.")

ValidationError: 1 validation error for TopologyGraph
nodes
  Input should be a valid dictionary or instance of TopologyNodes [type=model_type, input_value=TopologyNodes(servers=[Se..., ram_per_process=None)), input_type=TopologyNodes]
    For further information visit https://errors.pydantic.dev/2.11/v/model_type

## 3) MMc (Round-Robin) — theory vs observed comparison

If the payload violates MMc assumptions, a readable error is shown instead.
This section matches exactly what the analyzer computes **now**: a **split random model** (not the pooled FCFS/Erlang-C model). When we add **FCFS** in the LB, we’ll also expose the textbook Erlang-C formulas side-by-side.

---

## Variables (what they represent)

* **$c$**: number of *identical* servers (parallel replicas).
* **$\lambda$**: global **arrival rate** (req/s).
* **$\mu$**: **per-server** service rate (req/s) $= 1/\mathbb{E}[S]$.
* **$\rho$**: **global utilization**, $\rho = \lambda/(c\,\mu)$ (unitless).
* **$W$**: **mean time in system** (queue + service), seconds.
* **$W_q$**: **mean waiting time in queue**, seconds.
* **$L$**: **mean number in system** (queue + service), unitless.
* **$L_q$**: **mean number in queue**, unitless.
* **$\mathbb{E}[S]$**: mean **CPU service time**, seconds.

Derived (random split model):

* **$\lambda_i$**: **per-server arrival rate**, $\lambda_i = \lambda/c$.

> In the comparison table you’ll see two columns: **Theory** (closed-form) and **Observed** (estimates from the run).

---

## How we compute the **Theory** column (MMc with Round-Robin split)

1. **Predicted arrival rate**

$$
\lambda_{\text{Theory}} \;=\; 
\frac{\texttt{avg\_active\_users.mean}\times
      \texttt{avg\_request\_per\_minute\_per\_user.mean}}{60}
$$

2. **Predicted service rate** (from the **CPU exponential step**)

$$
\mu_{\text{Theory}} \;=\; \frac{1}{\mathbb{E}[S]}
$$

3. **Parallelism & utilization**

$$
c \;=\; \text{number of servers}, 
\qquad
\rho_{\text{Theory}} \;=\; \frac{\lambda_{\text{Theory}}}{c\,\mu_{\text{Theory}}}
$$

4. **RR split closed forms** (used by the analyzer today)

If $\rho_{\text{Theory}} \ge 1$: the system is **unstable** and
$W, W_q, L, L_q$ **diverge** (displayed as $+\infty$).

Otherwise, let $\lambda_i = \lambda_{\text{Theory}}/c$. We use:

$$
\begin{aligned}
W_{q,\text{Theory}} &= \frac{\rho_{\text{Theory}}}
                            {\mu_{\text{Theory}} - \lambda_i} \\
W_{\text{Theory}}   &= \frac{1}{\mu_{\text{Theory}}} + W_{q,\text{Theory}} \\
L_{q,\text{Theory}} &= \lambda_{\text{Theory}} \, W_{q,\text{Theory}} \\
L_{\text{Theory}}   &= \lambda_{\text{Theory}} \, W_{\text{Theory}}
\end{aligned}
$$

> 🔎 **Note:** These formulas reflect a **random split** into *c* identical M/M/1 queues (no central pool). They are **not** the Erlang-C (pooled FCFS) formulas. Once we add **FCFS** at the LB, we’ll surface the **textbook pooled M/M/c** KPIs (including $P_0$, $P_w$, etc.) alongside this random split model.

---

## How we compute the **Observed** column (from the run)

After `ResultsAnalyzer.process_all_metrics()`:

1. **Observed arrival rate** (system throughput)

$$
\lambda_{\text{Observed}} \;=\; 
\text{mean}\big(\text{windowed RPS series (client completions)}\big)
$$

*(This is end-to-end, i.e., what exits all servers and reaches the client.)*

2. **Observed time in system** (client E2E latency)

$$
W_{\text{Observed}} \;=\; \text{mean}\big(\text{client latencies}\big)
$$

3. **Observed service rate** (aggregate across servers)
   Let $\overline{S} = \text{mean}(\texttt{service\_time})$ *over all servers, weighted by number of jobs*:

$$
\mu_{\text{Observed}} \;=\;
\begin{cases}
1/\overline{S}, & \overline{S} > 0 \\
+\infty, & \overline{S} = 0
\end{cases}
$$

4. **Observed waiting time in queue** (aggregate across servers)

$$
W_{q,\text{Observed}} \;=\; 
\text{mean}\big(\texttt{waiting\_time}\big)
$$

5. **Little’s law (observed)**

$$
L_{\text{Observed}}=\lambda_{\text{Observed}}\, W_{\text{Observed}},
\qquad
L_{q,\text{Observed}}=\lambda_{\text{Observed}}\, W_{q,\text{Observed}}
$$

6. **Observed utilization (global)**

$$
\rho_{\text{Observed}} \;=\;
\begin{cases}
\lambda_{\text{Observed}} / (c\,\mu_{\text{Observed}}),
 & \mu_{\text{Observed}} \not\in \{0, +\infty\} \\
0, & \text{otherwise}
\end{cases}
$$

---

### Why small deltas appear

* **RR vs pooled:** We’re modeling **RR split**, not pooled FCFS; pooled Erlang-C typically yields different $W_q$ at the same $\lambda,\mu,c$.
* **Windowed arrivals:** The generator uses a **windowed** active-user sampler; across windows the rate is piecewise-constant (Cox process), not one global Poisson.
* **Finite horizon / warm-up:** Short runs and lack of warm-up bias early windows.
* **Network latency:** Small but non-zero link delays add a constant component to $W$.
* **Server identity:** Any heterogeneity breaks the “identical servers” assumption.

Increasing the run length, keeping edges tiny and deterministic, and ensuring identical servers will usually shrink Theory–Observed gaps. 

In [ ]:
mmc = MMc()
if mmc.is_compatible(payload):
   mmc.print_comparison(payload, results)  
else:
    print("Payload is not compatible with M/M/c:")
    for reason in mmc.explain_incompatibilities(payload):
        print(" -", reason)
   


MMc (RR) — Theory vs Observed
--------------------------------------------------------------------
sym  metric                    theory    observed        abs    rel%
--------------------------------------------------------------------
λ    Arrival rate (1/s)     40.000000   39.350000  -0.650000   -1.62
μ    Service rate (1/s)    100.000000  100.395641   0.395641    0.40
c    Servers                 2.000000    2.000000   0.000000    0.00
rho  Utilization             0.200000    0.195975  -0.004025   -2.01
L    Mean items in sys       0.500000    0.431651  -0.068349  -13.67
Lq   Mean items in queue     0.100000    0.038128  -0.061872  -61.87
W    Mean time in sys (s)    0.012500    0.010970  -0.001530  -12.24
Wq   Mean waiting (s)        0.002500    0.000969  -0.001531  -61.24
